In [18]:
import pandas as pd
import numpy as np
import re
import html
import os

In [19]:
df = pd.read_csv("../Data/Raw/IMDB Dataset.csv")
print("Dataset loaded successfully!")

Dataset loaded successfully!


In [20]:
df_clean = df.copy()

In [21]:
print("Dataset shape:", df_clean.shape)

Dataset shape: (50000, 2)


In [22]:
print("Columns:", df_clean.columns.tolist())

Columns: ['review', 'sentiment']


In [23]:
df_clean.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [24]:
df_clean.tail()

,review,sentiment
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative
49999,No one expects the Star Trek movies to be high...,negative


In [25]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [26]:
df_clean.describe(include="all")

,review,sentiment
count,50000,50000
unique,49582,2
top,Loved today's show!!! It was a variety and not...,positive
freq,5,25000


In [30]:
print(df_clean.isnull().sum())
missing_percentage = (
    df_clean.isnull().sum() / len(df_clean)
) * 100

print(missing_percentage)

review       0
sentiment    0
dtype: int64
review       0.0
sentiment    0.0
dtype: float64


In [31]:
empty_reviews = df_clean["review"].str.strip().eq("").sum()

empty_sentiments = df_clean["sentiment"].str.strip().eq("").sum()

print("Empty reviews:", empty_reviews)
print("Empty sentiments:", empty_sentiments)

Empty reviews: 0
Empty sentiments: 0


In [34]:
print("Duplicate rows:", df_clean.duplicated().sum())
df_clean[df_clean.duplicated(keep=False)].head(5)

Duplicate rows: 418


,review,sentiment
42,"Of all the films I have seen, this one, The Ra...",negative
84,"We brought this film as a joke for a friend, a...",negative
140,"Before I begin, let me get something off my ch...",negative
219,Ed Wood rides again. The fact that this movie ...,negative
245,I have seen this film at least 100 times and I...,positive


In [36]:
print(
    "Duplicate review texts:",
    df_clean["review"].duplicated().sum()
)
duplicate_reviews = df_clean[
    df_clean["review"].duplicated(keep=False)
]

conflicts = (
    duplicate_reviews.groupby("review")["sentiment"]
    .nunique()
)

print(
    "Reviews with conflicting labels:",
    (conflicts > 1).sum()
)

Duplicate review texts: 418
Reviews with conflicting labels: 0


In [37]:
df_clean = df_clean.drop_duplicates()

df_clean = df_clean.reset_index(drop=True)

print("Shape after removing duplicates:", df_clean.shape)

Shape after removing duplicates: (49582, 2)


In [38]:
print(df_clean.dtypes)

review       object
sentiment    object
dtype: object


In [39]:
print(df_clean["sentiment"].unique())

['positive' 'negative']


In [40]:
print(df_clean["sentiment"].value_counts())

sentiment
positive    24884
negative    24698
Name: count, dtype: int64


In [41]:
print(
    df_clean["sentiment"].value_counts(normalize=True) * 100
)

sentiment
positive    50.187568
negative    49.812432
Name: proportion, dtype: float64


In [42]:
df_clean["sentiment"] = (
    df_clean["sentiment"]
    .str.strip()
    .str.lower()
)

In [43]:
valid_labels = ["positive", "negative"]

invalid_labels = ~df_clean["sentiment"].isin(valid_labels)

print("Invalid labels:", invalid_labels.sum())

Invalid labels: 0


In [44]:
html_count = df_clean["review"].str.contains(
    r"<[^>]+>",
    regex=True
).sum()

print("Reviews containing HTML tags:", html_count)

Reviews containing HTML tags: 28968


In [45]:
def remove_html(text):

    # Replace HTML tags with spaces
    text = re.sub(r"<[^>]+>", " ", text)

    # Decode HTML entities
    text = html.unescape(text)

    return text

In [46]:
df_clean["review"] = df_clean["review"].apply(remove_html)

In [47]:
def clean_whitespace(text):

    # Remove spaces from beginning and end
    text = text.strip()

    # Replace multiple spaces with a single space
    text = re.sub(r"\s+", " ", text)

    return text

In [48]:
df_clean["review"] = df_clean["review"].apply(clean_whitespace)

In [49]:
entity_pattern = r"&(?:amp|lt|gt|quot|#39);"

print(
    "Common HTML entities remaining:",
    df_clean["review"].str.contains(
        entity_pattern,
        case=False,
        regex=True
    ).sum()
)

Common HTML entities remaining: 0


In [50]:
print(
    "Empty reviews after cleaning:",
    df_clean["review"].str.strip().eq("").sum()
)

Empty reviews after cleaning: 0


In [55]:
duplicates = df_clean[
    df_clean.duplicated(keep=False)
]

print("Remaining duplicate records:")
display(duplicates)

Remaining duplicate records:


,review,sentiment
7477,The first murder scene is one of the best murd...,positive
13469,Ah yet another Seagal movie.In no less than a ...,negative
14521,This movie got off to an interesting start. Do...,negative
22921,The first murder scene is one of the best murd...,positive
23801,Ah yet another Seagal movie.In no less than a ...,negative
27788,How Tasty Was My Little Frenchman tells a stor...,positive
44530,This movie got off to an interesting start. Do...,negative
47715,How Tasty Was My Little Frenchman tells a stor...,positive


In [56]:
duplicate_reviews = df_clean[
    df_clean["review"].duplicated(keep=False)
]

conflicts = (
    duplicate_reviews.groupby("review")["sentiment"]
    .nunique()
)

print("Conflicting sentiment labels:", (conflicts > 1).sum())

Conflicting sentiment labels: 0


In [57]:
df_clean = df_clean.drop_duplicates()

df_clean = df_clean.reset_index(drop=True)

print("Remaining duplicates:", df_clean.duplicated().sum())
print("Updated dataset shape:", df_clean.shape)

Remaining duplicates: 0
Updated dataset shape: (49578, 2)


In [58]:
print("FINAL DATASET REPORT")
print("--------------------")

print("Total records:", len(df_clean))

print("Total columns:", len(df_clean.columns))

print("\nMissing values:")
print(df_clean.isnull().sum())

print("\nDuplicate rows:", df_clean.duplicated().sum())

print(
    "\nEmpty reviews:",
    df_clean["review"].str.strip().eq("").sum()
)

print(
    "\nEmpty sentiments:",
    df_clean["sentiment"].str.strip().eq("").sum()
)

print(
    "\nInvalid sentiment labels:",
    (~df_clean["sentiment"].isin(
        ["positive", "negative"]
    )).sum()
)

print("\nSentiment distribution:")
print(df_clean["sentiment"].value_counts())

print("\nData types:")
print(df_clean.dtypes)

print("\nFinal dataset shape:", df_clean.shape)

FINAL DATASET REPORT
--------------------
Total records: 49578
Total columns: 2

Missing values:
review       0
sentiment    0
dtype: int64

Duplicate rows: 0

Empty reviews: 0

Empty sentiments: 0

Invalid sentiment labels: 0

Sentiment distribution:
sentiment
positive    24882
negative    24696
Name: count, dtype: int64

Data types:
review       object
sentiment    object
dtype: object

Final dataset shape: (49578, 2)


In [59]:
df_clean.to_csv(
    "../Data/Processed/cleaned_dataset.csv",
    index=False
)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!
